In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [4]:
import re  # regex to clean up newlines
import hashlib  # build a stable ID (fingerprint) from text
from pathlib import Path  # get file name from path

from langchain_community.document_loaders import PyPDFDirectoryLoader  # loads every PDF in a folder
from langchain_text_splitters import RecursiveCharacterTextSplitter  # splits text into small chunks
from langchain_ollama import OllamaEmbeddings, ChatOllama  # Ollama embedding and chat model
from langchain_chroma import Chroma  # vector database

# ---------- 1. Settings (one place, easy to change) ----------
CV_FOLDER = "data/cvs"  # path to the CV folder
CHUNK_SIZE = 800  # max ~800 characters per chunk
CHUNK_OVERLAP = 100  # ~100 character overlap with the neighboring chunk
COLLECTION = "cv_chunks"  # vector store collection name
EMBED_MODEL = "nomic-embed-text"  # model used for embeddings
CHAT_MODEL = "llama3.2:3b"  # model used to answer questions

/var/folders/fx/n2sq58254dsdkq7q30ldz5qw0000gn/T/ipykernel_6339/2486048563.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader  # loads every PDF in a folder
/opt/homebrew/Caskroom/miniconda/base/envs/langchain/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# ---------- 2. Load PDFs ----------
docs = PyPDFDirectoryLoader(CV_FOLDER, glob="**/*.pdf").load()  # one Document per page of each PDF

In [16]:

print(len(docs))
print(docs)

3
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'candidate': 'Ashikujjaman Shaikat Resume'}, page_content='Md Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Software Engineer with 5+ years of professional experience building scalable, responsive, and maintainable web applications using React, Vue, Next.js, JavaScript, and TypeScript. Experienced in developing reusable UI components, implementing state management, and translating UI/UX designs into accessible, pixel-perfect interfaces. Strong understanding of frontend architecture, performance optimization, API integration, and cross-functional collaboration with international teams. SKILLS SUMMARY ❖ Frontend Technology: ❖ UI & Styling: ❖ State Management: ❖ Testing & Code Quality: ❖ Architecture & Design: ❖ Backend & APIs: ❖ Databa

In [9]:
for d in docs:  # for every page
    d.page_content = re.sub(r"\s+", " ", d.page_content).strip()  # collapse all newlines/extra spaces into one space
    d.metadata["candidate"] = Path(d.metadata["source"]).stem  # file name without extension


In [14]:
print(len(docs))
print(docs)

3
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'candidate': 'Ashikujjaman Shaikat Resume'}, page_content='Md Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Software Engineer with 5+ years of professional experience building scalable, responsive, and maintainable web applications using React, Vue, Next.js, JavaScript, and TypeScript. Experienced in developing reusable UI components, implementing state management, and translating UI/UX designs into accessible, pixel-perfect interfaces. Strong understanding of frontend architecture, performance optimization, API integration, and cross-functional collaboration with international teams. SKILLS SUMMARY ❖ Frontend Technology: ❖ UI & Styling: ❖ State Management: ❖ Testing & Code Quality: ❖ Architecture & Design: ❖ Backend & APIs: ❖ Databa

In [11]:
# ---------- 4. Split ----------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,  # max chunk size
    chunk_overlap=CHUNK_OVERLAP,  # overlap
    is_separator_regex=True,  # the separators below are regex
    separators=[
        r"\s+(?=●)",  # split before ● (bullet stays with the text after it)
        r"(?<=[.!?])\s+",  # split at end of sentence
        r" ",  # split at word boundaries
        r"",  # last resort: split by character
    ],
)
chunks = splitter.split_documents(docs) 

In [18]:
print(len(chunks), "chunks created from", len(docs), "pages of CVs")
print(chunks)

11 chunks created from 3 pages of CVs
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'candidate': 'Ashikujjaman Shaikat Resume'}, page_content='Md Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Software Engineer with 5+ years of professional experience building scalable, responsive, and maintainable web applications using React, Vue, Next.js, JavaScript, and TypeScript. Experienced in developing reusable UI components, implementing state management, and translating UI/UX designs into accessible, pixel-perfect interfaces. Strong understanding of frontend architecture, performance optimization, API integration, and cross-functional collaboration with international teams.'), Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'ti

In [19]:
# ---------- 5. Attach candidate name to every chunk ----------
for c in chunks:
    c.page_content = f"CV of {c.metadata['candidate']}:\n{c.page_content}"  # so the LLM knows whose CV this is

In [24]:
print(len(chunks), "chunks created from", len(docs), "pages of CVs")
print(chunks)

11 chunks created from 3 pages of CVs
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'candidate': 'Ashikujjaman Shaikat Resume'}, page_content='CV of Ashikujjaman Shaikat Resume:\nMd Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Software Engineer with 5+ years of professional experience building scalable, responsive, and maintainable web applications using React, Vue, Next.js, JavaScript, and TypeScript. Experienced in developing reusable UI components, implementing state management, and translating UI/UX designs into accessible, pixel-perfect interfaces. Strong understanding of frontend architecture, performance optimization, API integration, and cross-functional collaboration with international teams.'), Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creato

In [28]:
%pip install langchain-pinecone pinecone

Note: you may need to restart the kernel to use updated packages.


In [29]:
from pinecone import Pinecone, ServerlessSpec
import getpass
import os
import time

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

In [30]:
import time

index_name = "cv-search-index"  # change if desired

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)

In [33]:
pc.describe_index(index_name)

{
    "name": "cv-search-index",
    "metric": "cosine",
    "host": "cv-search-index-7qv6k7v.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [31]:
%pip install langchain_huggingface

Note: you may need to restart the kernel to use updated packages.


In [32]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8646.66it/s]


In [34]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embedding_model)

In [35]:
vector_store.add_documents(documents=chunks)

['43f0a00b-0110-4db7-90ba-0bc7d0271c52',
 'cc7bfa44-0030-40fc-bd54-5d06a89c5af3',
 '30cf1c4c-9ed9-480a-8671-aa131a98d14e',
 '706abd06-34b8-49c0-88bd-54084d7e0b0f',
 '4856b3f4-eddc-4638-8f80-582d6a795f82',
 '23aae701-8997-4843-8cef-f4e9ce2abe5e',
 'f4cccb98-9aa1-4b31-8cef-af1fee7723d2',
 '9af8ad0f-5c5d-4457-a7af-79ba1a9ca110',
 '4aa08094-fdd1-4790-8ae4-8d91a2fe4368',
 '4f20bf78-45fe-4562-9110-aaea5aa9f042',
 '3bd7853a-95c2-44b1-8d37-076ab77c7fb5']

In [36]:
pc.list_indexes()

[
    {
        "name": "cv-search-index",
        "metric": "cosine",
        "host": "cv-search-index-7qv6k7v.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "langchain-test-index",
        "metric": "cosine",
        "host": "langchain-test-index-7qv6k7v.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": 

In [40]:
query = "what is the use case being discussed in the documents?"
results = vector_store.similarity_search(query, k=3)

print("\n How many years of experience?")
for idx, result in enumerate(results, start=1):
    print(f"{idx}. {result.page_content}")


 How many years of experience?
1. CV of Ashikujjaman Shaikat Resume:
- Integrated RESTful APIs with frontend applications to display and process banking data and business transactions. - Implemented JWT-based authentication, role-based access control (RBAC), and secure user-facing workflows. - Translated business requirements and UI/UX designs into responsive, accessible, and pixel-perfect interfaces. - Collaborated with backend engineers working with Java, Spring Boot, Hibernate, and Oracle to ensure seamless frontend and backend integration. - Maintained code quality through reusable components, debugging, code reviews, and frontend testing. N.B.: I cannot disclose the technology or business requirements for my projects as they are proprietary to BJIT. EDUCATION Daffodil International University B.Sc.
2. CV of Ashikujjaman Shaikat Resume:
- Reusable UI Components: Design and maintain reusable UI components and shared utilities using Atomic Design, BEM, and component-based architectu

In [41]:
results = vector_store.similarity_search_with_score(
    "what is use case being discussed in the documents?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.225608] CV of Ashikujjaman Shaikat Resume:
- Integrated RESTful APIs with frontend applications to display and process banking data and business transactions. - Implemented JWT-based authentication, role-based access control (RBAC), and secure user-facing workflows. - Translated business requirements and UI/UX designs into responsive, accessible, and pixel-perfect interfaces. - Collaborated with backend engineers working with Java, Spring Boot, Hibernate, and Oracle to ensure seamless frontend and backend integration. - Maintained code quality through reusable components, debugging, code reviews, and frontend testing. N.B.: I cannot disclose the technology or business requirements for my projects as they are proprietary to BJIT. EDUCATION Daffodil International University B.Sc. [{'candidate': 'Ashikujjaman Shaikat Resume', 'creationdate': '', 'creator': 'PyPDF', 'page': 1.0, 'page_label': '2', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'source': 'data/cvs/Ashikujjaman S

In [58]:
retriever = vector_store.as_retriever()
retriever_result = retriever.invoke("How many years of experience?")

In [64]:
retriever_result

[Document(id='30cf1c4c-9ed9-480a-8671-aa131a98d14e', metadata={'candidate': 'Ashikujjaman Shaikat Resume', 'creationdate': '', 'creator': 'PyPDF', 'page': 0.0, 'page_label': '1', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'title': 'Ashikujjaman Shaikat Resume', 'total_pages': 3.0}, page_content='CV of Ashikujjaman Shaikat Resume:\nBJIT Limited ( Job Title: Senior Software Engineer ) April 2022 ‑ Present ( 5+ years ) Role: Collaboration with Japanese and Philippine Teams (3 years) Core Responsibilities & Technical Expertise - Frontend Engineering: Develop scalable, responsive, and maintainable web applications using React.js, Next.js, JavaScript, and TypeScript.'),
 Document(id='cc7bfa44-0030-40fc-bd54-5d06a89c5af3', metadata={'candidate': 'Ashikujjaman Shaikat Resume', 'creationdate': '', 'creator': 'PyPDF', 'page': 0.0, 'page_label': '1', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'source': 'data/cvs/Ashikujjaman Shai

In [65]:
retriever = vector_store.as_retriever(search_type="mmr")

In [66]:
mmr_retriever_result = retriever.invoke("How many years of experience?")

In [67]:
mmr_retriever_result

[Document(metadata={'candidate': 'Ashikujjaman Shaikat Resume', 'creationdate': '', 'creator': 'PyPDF', 'page': 0.0, 'page_label': '1', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'title': 'Ashikujjaman Shaikat Resume', 'total_pages': 3.0}, page_content='CV of Ashikujjaman Shaikat Resume:\nBJIT Limited ( Job Title: Senior Software Engineer ) April 2022 ‑ Present ( 5+ years ) Role: Collaboration with Japanese and Philippine Teams (3 years) Core Responsibilities & Technical Expertise - Frontend Engineering: Develop scalable, responsive, and maintainable web applications using React.js, Next.js, JavaScript, and TypeScript.'),
 Document(metadata={'candidate': 'Ashikujjaman Shaikat Resume', 'creationdate': '', 'creator': 'PyPDF', 'page': 2.0, 'page_label': '3', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'title': 'Ashikujjaman Shaikat Resume', 'total_pages': 3.0}, page_con

In [88]:
retriever = vector_store.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.7})

In [89]:
scr_threshold_docs = retriever.invoke("How many years of experience?")

In [90]:
scr_threshold_docs

[Document(id='30cf1c4c-9ed9-480a-8671-aa131a98d14e', metadata={'candidate': 'Ashikujjaman Shaikat Resume', 'creationdate': '', 'creator': 'PyPDF', 'page': 0.0, 'page_label': '1', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'source': 'data/cvs/Ashikujjaman Shaikat Resume.pdf', 'title': 'Ashikujjaman Shaikat Resume', 'total_pages': 3.0}, page_content='CV of Ashikujjaman Shaikat Resume:\nBJIT Limited ( Job Title: Senior Software Engineer ) April 2022 ‑ Present ( 5+ years ) Role: Collaboration with Japanese and Philippine Teams (3 years) Core Responsibilities & Technical Expertise - Frontend Engineering: Develop scalable, responsive, and maintainable web applications using React.js, Next.js, JavaScript, and TypeScript.')]